# Smog Forecasting Debug Workflow
This notebook is intentionally split into isolated stages so you can identify where failures happen.

Stages covered:
1. Environment and path setup
2. Raw dataset inspection
3. Sentinel-5P preprocessing
4. MERRA-2 preprocessing and alignment
5. Fusion and dataset validation
6. Data generator sanity checks
7. ConvLSTM model smoke test
8. Optional dry-run training
9. Optional evaluation

> Heavy steps are guarded by flags. Run cells from top to bottom and only enable the expensive sections when the previous stage is clean.

In [16]:
from pathlib import Path
import json
import sys
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import xarray as xr


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'Dataset').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root containing both Dataset/ and src/.')


ROOT_DIR = find_project_root(Path.cwd())
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Project root: {ROOT_DIR}')
print(f'Python executable: {sys.executable}')
print(f'TensorFlow version: {tf.__version__}')

Project root: C:\Users\Pranjal\Documents\GitHub\fifth-season
Python executable: c:\Users\Pranjal\miniconda3\envs\tf\python.exe
TensorFlow version: 2.10.1


In [17]:
# ── GPU / CUDA configuration ──────────────────────────────────────────────────
# RTX 5060 (compute capability 12.0) requires PTX JIT compilation on first run.
# Memory growth prevents TF from allocating all VRAM upfront.
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # suppress verbose CUDA info messages

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s) enabled with memory growth: {[g.name for g in gpus]}')
else:
    print('No GPU detected — running on CPU.')

print(f'TF built with CUDA: {tf.test.is_built_with_cuda()}')

GPU(s) enabled with memory growth: ['/physical_device:GPU:0']
TF built with CUDA: True


## 1. Parameters and Configuration

In [18]:
from data.process_sentinel import process_sentinel
from data.process_merra2 import process_merra2
from data.fuse_datasets import fuse_datasets
from model.convlstm import compile_mtl_unet_convlstm
from train import WeatherDataGenerator, _load_dataset_config, train_model
from evaluate import evaluate_model

DATASET_DIR = ROOT_DIR / 'Dataset'
PROCESSED_DIR = DATASET_DIR / 'processed'
ARTIFACT_DIR = ROOT_DIR / 'artifacts' / 'notebook_training'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SENTINEL_SOURCE = DATASET_DIR / 'S5PL2_5D.nc'
MERRA_DIR = DATASET_DIR / 'merra-2-till-DEC2025'
SENTINEL_PROCESSED = PROCESSED_DIR / 'sentinel_5d.nc'
MERRA_PROCESSED = PROCESSED_DIR / 'merra2_5d_aligned.nc'
FUSED_PATH = PROCESSED_DIR / 'fused_5D_debug.nc'
TRAIN_PATH = PROCESSED_DIR / 'train_5D_debug.nc'
TEST_PATH = PROCESSED_DIR / 'test_5D_debug.nc'

SEQUENCE_LENGTH = 5
BATCH_SIZE = 1
LEARNING_RATE = 1e-5
DRY_RUN_MAX_SAMPLES = 10
DRY_RUN_EPOCHS = 2
FULL_TRAIN_EPOCHS = 20
FULL_TRAIN_MAX_SAMPLES = None
EVALUATION_MAX_SAMPLES = None
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_MONITOR = 'val_loss'
SMOKE_TEST_HEIGHT = 64
SMOKE_TEST_WIDTH = 64

# ── Set False for any stage whose output already exists on disk ───────────────
RUN_SENTINEL_PROCESSING = False
RUN_MERRA_PROCESSING = False
RUN_FUSION = False
BUILD_FULL_RES_MODEL = False
RUN_DRY_RUN_TRAINING = False
RUN_FULL_TRAINING = False   # set True to start a new training run
RUN_EVALUATION = True

print('Runtime flags:')
print({
    'RUN_SENTINEL_PROCESSING': RUN_SENTINEL_PROCESSING,
    'RUN_MERRA_PROCESSING': RUN_MERRA_PROCESSING,
    'RUN_FUSION': RUN_FUSION,
    'BUILD_FULL_RES_MODEL': BUILD_FULL_RES_MODEL,
    'RUN_DRY_RUN_TRAINING': RUN_DRY_RUN_TRAINING,
    'RUN_FULL_TRAINING': RUN_FULL_TRAINING,
    'RUN_EVALUATION': RUN_EVALUATION,
})

print('Training configuration:')
print({
    'DRY_RUN_EPOCHS': DRY_RUN_EPOCHS,
    'FULL_TRAIN_EPOCHS': FULL_TRAIN_EPOCHS,
    'EARLY_STOPPING_PATIENCE': EARLY_STOPPING_PATIENCE,
    'EARLY_STOPPING_MONITOR': EARLY_STOPPING_MONITOR,
    'BATCH_SIZE': BATCH_SIZE,
    'LEARNING_RATE': LEARNING_RATE,
})

print('Output targets:')
print({
    'ARTIFACT_DIR': str(ARTIFACT_DIR),
    'FUSED_PATH': str(FUSED_PATH),
    'TRAIN_PATH': str(TRAIN_PATH),
    'TEST_PATH': str(TEST_PATH),
})


Runtime flags:
{'RUN_SENTINEL_PROCESSING': False, 'RUN_MERRA_PROCESSING': False, 'RUN_FUSION': False, 'BUILD_FULL_RES_MODEL': False, 'RUN_DRY_RUN_TRAINING': False, 'RUN_FULL_TRAINING': False, 'RUN_EVALUATION': True}
Training configuration:
{'DRY_RUN_EPOCHS': 2, 'FULL_TRAIN_EPOCHS': 20, 'EARLY_STOPPING_PATIENCE': 4, 'EARLY_STOPPING_MONITOR': 'val_loss', 'BATCH_SIZE': 1, 'LEARNING_RATE': 1e-05}
Output targets:
{'ARTIFACT_DIR': 'C:\\Users\\Pranjal\\Documents\\GitHub\\fifth-season\\artifacts\\notebook_training', 'FUSED_PATH': 'C:\\Users\\Pranjal\\Documents\\GitHub\\fifth-season\\Dataset\\processed\\fused_5D_debug.nc', 'TRAIN_PATH': 'C:\\Users\\Pranjal\\Documents\\GitHub\\fifth-season\\Dataset\\processed\\train_5D_debug.nc', 'TEST_PATH': 'C:\\Users\\Pranjal\\Documents\\GitHub\\fifth-season\\Dataset\\processed\\test_5D_debug.nc'}


In [19]:
print('Current runtime flag snapshot:')
print({
    'RUN_SENTINEL_PROCESSING': RUN_SENTINEL_PROCESSING,
    'RUN_MERRA_PROCESSING': RUN_MERRA_PROCESSING,
    'RUN_FUSION': RUN_FUSION,
    'BUILD_FULL_RES_MODEL': BUILD_FULL_RES_MODEL,
    'RUN_DRY_RUN_TRAINING': RUN_DRY_RUN_TRAINING,
    'RUN_FULL_TRAINING': RUN_FULL_TRAINING,
    'RUN_EVALUATION': RUN_EVALUATION,
})

print('Current training snapshot:')
print({
    'DRY_RUN_EPOCHS': DRY_RUN_EPOCHS,
    'FULL_TRAIN_EPOCHS': FULL_TRAIN_EPOCHS,
    'EARLY_STOPPING_PATIENCE': EARLY_STOPPING_PATIENCE,
    'EARLY_STOPPING_MONITOR': EARLY_STOPPING_MONITOR,
})

print(f'MERRA processed file exists: {MERRA_PROCESSED.exists()} -> {MERRA_PROCESSED}')
print(f'Train dataset exists: {TRAIN_PATH.exists()} -> {TRAIN_PATH}')
print(f'Test dataset exists: {TEST_PATH.exists()} -> {TEST_PATH}')

Current runtime flag snapshot:
{'RUN_SENTINEL_PROCESSING': False, 'RUN_MERRA_PROCESSING': False, 'RUN_FUSION': False, 'BUILD_FULL_RES_MODEL': False, 'RUN_DRY_RUN_TRAINING': False, 'RUN_FULL_TRAINING': False, 'RUN_EVALUATION': True}
Current training snapshot:
{'DRY_RUN_EPOCHS': 2, 'FULL_TRAIN_EPOCHS': 20, 'EARLY_STOPPING_PATIENCE': 4, 'EARLY_STOPPING_MONITOR': 'val_loss'}
MERRA processed file exists: True -> C:\Users\Pranjal\Documents\GitHub\fifth-season\Dataset\processed\merra2_5d_aligned.nc
Train dataset exists: True -> C:\Users\Pranjal\Documents\GitHub\fifth-season\Dataset\processed\train_5D_debug.nc
Test dataset exists: True -> C:\Users\Pranjal\Documents\GitHub\fifth-season\Dataset\processed\test_5D_debug.nc


In [20]:
import importlib

import data.process_sentinel as process_sentinel_module
import data.process_merra2 as process_merra2_module
import data.fuse_datasets as fuse_datasets_module
import model.convlstm as convlstm_module
import train as train_module
import evaluate as evaluate_module

importlib.reload(process_sentinel_module)
importlib.reload(process_merra2_module)
importlib.reload(fuse_datasets_module)
importlib.reload(convlstm_module)
importlib.reload(train_module)
importlib.reload(evaluate_module)

process_sentinel = process_sentinel_module.process_sentinel
process_merra2 = process_merra2_module.process_merra2
fuse_datasets = fuse_datasets_module.fuse_datasets
compile_mtl_unet_convlstm = convlstm_module.compile_mtl_unet_convlstm
WeatherDataGenerator = train_module.WeatherDataGenerator
_load_dataset_config = train_module._load_dataset_config
train_model = train_module.train_model
evaluate_model = evaluate_module.evaluate_model

print('Reloaded project modules from disk.')

Reloaded project modules from disk.


## 2. Raw Dataset Inspection
Inspect the raw Sentinel-5P and MERRA-2 inputs before any processing. If this stage fails, the issue is with source files, dimensions, or timestamps rather than the model.

In [21]:
def preview_dataset(path: Path, label: str, max_vars: int = 12) -> None:
    path = Path(path)
    if not path.exists():
        print(f'{label}: missing -> {path}')
        return

    with xr.open_dataset(path) as ds:
        print(f'\n[{label}]')
        print(f'Path: {path}')
        print(f'Dimensions: {dict(ds.sizes)}')
        print(f'Variables ({len(ds.data_vars)} total): {list(ds.data_vars)[:max_vars]}')
        if 'time' in ds.coords and ds.sizes.get('time', 0) > 0:
            times = pd.to_datetime(ds['time'].values)
            print(f'Time range: {times.min()} -> {times.max()} ({len(times)} steps)')
        if 'lat' in ds.coords:
            print(f'Latitude range: {float(ds.lat.min())} -> {float(ds.lat.max())}')
        if 'lon' in ds.coords:
            print(f'Longitude range: {float(ds.lon.min())} -> {float(ds.lon.max())}')


preview_dataset(SENTINEL_SOURCE, 'Raw Sentinel-5P source')

merra_files = sorted(MERRA_DIR.glob('*.nc'))
print(f'\nMERRA-2 daily files found: {len(merra_files)}')
if merra_files:
    print('Sample files:')
    for sample_path in merra_files[:5]:
        print(f'  - {sample_path.name}')
    preview_dataset(merra_files[0], 'First raw MERRA-2 file')
else:
    print('No MERRA-2 NetCDF files were found in the source directory.')



[Raw Sentinel-5P source]
Path: C:\Users\Pranjal\Documents\GitHub\fifth-season\Dataset\S5PL2_5D.nc
Dimensions: {'time': 366, 'lat': 291, 'lon': 512, 'bnds': 2}
Variables (9 total): ['AER_AI_340_380', 'AER_AI_354_388', 'CH4', 'CLOUD_FRACTION', 'CO', 'HCHO', 'NO2', 'O3', 'SO2']
Time range: 2019-01-03 12:00:00 -> 2024-01-02 12:00:00 (366 steps)
Latitude range: 24.902743831054686 -> 34.361285842773434
Longitude range: 68.1535148310547 -> 84.82011816894531

MERRA-2 daily files found: 2693
Sample files:
  - MERRA2_400.tavg1_2d_aer_Nx.20190101.SUB (1).nc
  - MERRA2_400.tavg1_2d_aer_Nx.20190101.SUB.nc
  - MERRA2_400.tavg1_2d_aer_Nx.20190102.SUB (1).nc
  - MERRA2_400.tavg1_2d_aer_Nx.20190102.SUB.nc
  - MERRA2_400.tavg1_2d_aer_Nx.20190103.SUB (1).nc

[First raw MERRA-2 file]
Path: C:\Users\Pranjal\Documents\GitHub\fifth-season\Dataset\merra-2-till-DEC2025\MERRA2_400.tavg1_2d_aer_Nx.20190101.SUB (1).nc
Dimensions: {'time': 2, 'lon': 53, 'lat': 38}
Variables (5 total): ['BCCMASS', 'DUCMASS', 'OCCM

## 3. Process Sentinel-5P and MERRA-2 Separately
Run the processing modules one at a time. This makes it obvious whether failures come from Sentinel cleanup, MERRA-2 ingestion, interpolation, or the 5-day temporal alignment.

In [22]:
def describe_processed_dataset(path: Path, label: str) -> None:
    path = Path(path)
    if not path.exists():
        print(f'{label}: missing -> {path}')
        return

    with xr.open_dataset(path) as ds:
        print(f'\n[{label}]')
        print(f'Dimensions: {dict(ds.sizes)}')
        print(f'Variables ({len(ds.data_vars)} total): {list(ds.data_vars)[:10]}')
        if 'feature_variables' in ds.attrs:
            feature_variables = json.loads(ds.attrs['feature_variables'])
            print(f'Feature variable count: {len(feature_variables)}')
            print(f'Feature preview: {feature_variables[:8]}')


if RUN_SENTINEL_PROCESSING:
    try:
        process_sentinel(
            source_path=SENTINEL_SOURCE,
            output_path=SENTINEL_PROCESSED,
        )
        print(f'Sentinel processing completed: {SENTINEL_PROCESSED}')
    except Exception as exc:
        print('Sentinel processing failed.')
        traceback.print_exc()
        raise exc
else:
    print('Skipping Sentinel processing. Set RUN_SENTINEL_PROCESSING = True to regenerate the file.')

describe_processed_dataset(SENTINEL_PROCESSED, 'Processed Sentinel dataset')


Skipping Sentinel processing. Set RUN_SENTINEL_PROCESSING = True to regenerate the file.

[Processed Sentinel dataset]
Dimensions: {'time': 366, 'lat': 291, 'lon': 512}
Variables (9 total): ['AER_AI_340_380', 'AER_AI_354_388', 'CH4', 'CLOUD_FRACTION', 'CO', 'HCHO', 'NO2', 'O3', 'SO2']
Feature variable count: 8
Feature preview: ['AER_AI_354_388', 'CH4', 'CLOUD_FRACTION', 'CO', 'HCHO', 'NO2', 'O3', 'SO2']


### 3a. MERRA-2 Processing
Keep this in its own cell so interpolation, deduplication, and 5-day alignment errors are isolated from Sentinel preprocessing.

In [23]:
if RUN_MERRA_PROCESSING:
    try:
        process_merra2(
            merra_dir=MERRA_DIR,
            sentinel_reference_path=SENTINEL_PROCESSED,
            output_path=MERRA_PROCESSED,
        )
        print(f'MERRA-2 processing completed: {MERRA_PROCESSED}')
    except Exception as exc:
        print('MERRA-2 processing failed.')
        traceback.print_exc()
        raise exc
else:
    print('Skipping MERRA-2 processing. Set RUN_MERRA_PROCESSING = True once Sentinel processing is clean.')

describe_processed_dataset(MERRA_PROCESSED, 'Processed MERRA-2 dataset')

Skipping MERRA-2 processing. Set RUN_MERRA_PROCESSING = True once Sentinel processing is clean.

[Processed MERRA-2 dataset]
Dimensions: {'time': 366, 'lat': 291, 'lon': 512}
Variables (5 total): ['BCCMASS', 'DUCMASS', 'OCCMASS', 'SO4CMASS', 'SSCMASS']
Feature variable count: 5
Feature preview: ['BCCMASS', 'DUCMASS', 'OCCMASS', 'SO4CMASS', 'SSCMASS']


## 4. Fusion and Dataset Validation
This stage checks whether the processed Sentinel and MERRA-2 datasets align in time and space, then verifies that the final training tensors are finite and shaped correctly before any model code runs.

In [24]:
def summarize_fused_dataset(path: Path, label: str) -> None:
    path = Path(path)
    if not path.exists():
        print(f'{label}: missing -> {path}')
        return

    with xr.open_dataset(path) as ds:
        print(f'\n[{label}]')
        print(f'Dimensions: {dict(ds.sizes)}')
        print(f'Variables ({len(ds.data_vars)} total): {list(ds.data_vars)[:12]}')
        print(f'Target variable: {ds.attrs.get("target_variable", "<missing>")}')
        feature_variables = json.loads(ds.attrs.get('feature_variables', '[]'))
        print(f'Feature count: {len(feature_variables)}')
        if feature_variables:
            print(f'Feature preview: {feature_variables[:8]}')
        if ds.sizes.get('time', 0) > 0:
            times = pd.to_datetime(ds['time'].values)
            print(f'Time range: {times.min()} -> {times.max()} ({len(times)} steps)')
        else:
            print('Warning: dataset has zero timesteps. This artifact is not usable for training.')


if RUN_FUSION:
    try:
        fused_dataset, train_dataset, test_dataset = fuse_datasets(
            sentinel_path=SENTINEL_PROCESSED,
            merra_path=MERRA_PROCESSED,
            fused_output_path=FUSED_PATH,
            train_output_path=TRAIN_PATH,
            test_output_path=TEST_PATH,
        )
        print('Fusion completed successfully.')
        print(f'Fused: {dict(fused_dataset.sizes)}')
        print(f'Train: {dict(train_dataset.sizes)}')
        print(f'Test: {dict(test_dataset.sizes)}')
    except Exception as exc:
        print('Fusion failed.')
        traceback.print_exc()
        raise exc
else:
    print('Skipping fusion. Set RUN_FUSION = True after both processed files look correct.')

summarize_fused_dataset(FUSED_PATH, 'Fused dataset')
summarize_fused_dataset(TRAIN_PATH, 'Train dataset')
summarize_fused_dataset(TEST_PATH, 'Test dataset')


Skipping fusion. Set RUN_FUSION = True after both processed files look correct.

[Fused dataset]
Dimensions: {'time': 366, 'lat': 291, 'lon': 512}
Variables (15 total): ['AER_AI_354_388', 'CH4', 'CLOUD_FRACTION', 'CO', 'HCHO', 'NO2', 'O3', 'SO2', 'AER_AI_340_380', 'BCCMASS', 'DUCMASS', 'OCCMASS']
Target variable: AER_AI_340_380
Feature count: 13
Feature preview: ['AER_AI_354_388', 'CH4', 'CLOUD_FRACTION', 'CO', 'HCHO', 'NO2', 'O3', 'SO2']
Time range: 2019-01-03 12:00:00 -> 2024-01-02 12:00:00 (366 steps)

[Train dataset]
Dimensions: {'time': 292, 'lat': 291, 'lon': 512}
Variables (15 total): ['AER_AI_354_388', 'CH4', 'CLOUD_FRACTION', 'CO', 'HCHO', 'NO2', 'O3', 'SO2', 'AER_AI_340_380', 'BCCMASS', 'DUCMASS', 'OCCMASS']
Target variable: AER_AI_340_380
Feature count: 13
Feature preview: ['AER_AI_354_388', 'CH4', 'CLOUD_FRACTION', 'CO', 'HCHO', 'NO2', 'O3', 'SO2']
Time range: 2019-01-03 12:00:00 -> 2022-12-28 12:00:00 (292 steps)

[Test dataset]
Dimensions: {'time': 73, 'lat': 291, 'lon': 

### 4a. Data Generator Sanity Check
Use this cell to confirm that the fused training file produces valid ConvLSTM input windows and target tensors before building the model.

In [25]:
if TRAIN_PATH.exists():
    feature_variables, target_variable, num_classes = _load_dataset_config(TRAIN_PATH)
    print(f'Loaded training config: target={target_variable}, features={len(feature_variables)}, classes={num_classes}')

    generator = WeatherDataGenerator(
        TRAIN_PATH,
        feature_variables=feature_variables,
        target_variable=target_variable,
        sequence_length=SEQUENCE_LENGTH,
        batch_size=BATCH_SIZE,
        shuffle=False,
        max_samples=DRY_RUN_MAX_SAMPLES,
    )
    print(f'Generator batches available: {len(generator)}')

    if len(generator) > 0:
        batch_x, batch_y = generator[0]
        print(f'Input batch shape: {batch_x.shape}')
        print(f'Severity target shape: {batch_y["severity_output"].shape}')
        print(f'Identity target shape: {batch_y["identity_output"].shape}')
        print(f'Input finite: {np.isfinite(batch_x).all()}')
        print(f'Severity finite: {np.isfinite(batch_y["severity_output"]).all()}')
        print(f'Identity classes present: {np.unique(batch_y["identity_output"])[:10]}')
    else:
        print('Generator has no valid forecasting windows yet. Check the time dimension and sequence length.')
else:
    print('Training dataset is not available yet, so the generator sanity check was skipped.')

Loaded training config: target=AER_AI_340_380, features=13, classes=5
Generator batches available: 10
Input batch shape: (1, 5, 291, 512, 13)
Severity target shape: (1, 1, 291, 512, 1)
Identity target shape: (1, 1, 291, 512)
Input finite: True
Severity finite: True
Identity classes present: [1]


## 5. Model Smoke Test and Optional Dry-Run Training
Build the ConvLSTM model only after the fused dataset and generator look correct. The first pass uses a cropped grid so you can detect layer or loss shape errors without immediately hitting GPU or RAM limits.

In [26]:
history = None
smoke_model = None
full_model = None

if TRAIN_PATH.exists():
    feature_variables, target_variable, num_classes = _load_dataset_config(TRAIN_PATH)
    preview_generator = WeatherDataGenerator(
        TRAIN_PATH,
        feature_variables=feature_variables,
        target_variable=target_variable,
        sequence_length=SEQUENCE_LENGTH,
        batch_size=1,
        shuffle=False,
        max_samples=1,
    )

    if len(preview_generator) == 0:
        print('No valid training windows are available yet, so the model smoke test was skipped.')
    else:
        batch_x, batch_y = preview_generator[0]
        smoke_x = batch_x[:, :, :SMOKE_TEST_HEIGHT, :SMOKE_TEST_WIDTH, :]
        smoke_y = {
            'severity_output': tf.convert_to_tensor(
                batch_y['severity_output'][:, :, :SMOKE_TEST_HEIGHT, :SMOKE_TEST_WIDTH, :],
                dtype=tf.float32,
            ),
            'identity_output': tf.convert_to_tensor(
                batch_y['identity_output'][:, :, :SMOKE_TEST_HEIGHT, :SMOKE_TEST_WIDTH],
                dtype=tf.int32,
            ),
        }

        print(f'Smoke test input shape: {smoke_x.shape}')
        smoke_model = compile_mtl_unet_convlstm(
            input_shape=smoke_x.shape[1:],
            num_classes=num_classes,
            learning_rate=LEARNING_RATE,
        )
        smoke_metrics = smoke_model.evaluate(smoke_x, smoke_y, verbose=0, return_dict=True)
        print('Smoke test metrics:')
        print(smoke_metrics)

        if BUILD_FULL_RES_MODEL:
            print('Building the full-resolution model. This may consume substantial memory.')
            full_model = compile_mtl_unet_convlstm(
                input_shape=batch_x.shape[1:],
                num_classes=num_classes,
                learning_rate=LEARNING_RATE,
            )
            full_model.summary()
        else:
            print('Skipping full-resolution model build. Set BUILD_FULL_RES_MODEL = True after the cropped smoke test is clean.')
else:
    print('Training dataset is missing. Complete fusion before attempting model checks.')

Smoke test input shape: (1, 5, 64, 64, 13)
Smoke test metrics:
{'loss': 2.011580467224121, 'severity_output_loss': 0.5003557205200195, 'identity_output_loss': 1.010869026184082, 'severity_output_mse': 0.14920276403427124, 'severity_output_average_ssim': 0.4118560552597046, 'identity_output_accuracy': 0.143798828125}
Skipping full-resolution model build. Set BUILD_FULL_RES_MODEL = True after the cropped smoke test is clean.


### 5a. Optional 2-Epoch Dry-Run Training
Keep this short. Its only job is to verify the generator, losses, callbacks, and checkpoint path before you spend time on real training.

In [27]:
if RUN_DRY_RUN_TRAINING:
    try:
        _, history = train_model(
            train_dataset_path=TRAIN_PATH,
            val_dataset_path=TEST_PATH,
            output_dir=ARTIFACT_DIR,
            sequence_length=SEQUENCE_LENGTH,
            batch_size=BATCH_SIZE,
            epochs=DRY_RUN_EPOCHS,
            learning_rate=LEARNING_RATE,
            max_samples=DRY_RUN_MAX_SAMPLES,
            dry_run=True,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_monitor=EARLY_STOPPING_MONITOR,
        )
        print('Dry-run training completed.')
        print(f'History keys: {list(history.history.keys())}')
    except Exception as exc:
        print('Dry-run training failed.')
        traceback.print_exc()
        raise exc
else:
    print('Dry-run training is disabled. Keep this off once the pipeline is validated.')

Dry-run training is disabled. Keep this off once the pipeline is validated.


### 5b. Full Training Run
This is the real training stage. Start with 20 epochs, keep early stopping on `val_loss`, and inspect `val_severity_output_mse` and `val_severity_output_average_ssim` after training.

Each run is saved to a timestamped subdirectory inside the artifact folder (e.g. `run_20260310_143022/`) so no past experiments are ever overwritten. The best model is also copied to `best_model.keras` in the root for evaluation.


In [28]:
if RUN_FULL_TRAINING:
    try:
        _, history = train_model(
            train_dataset_path=TRAIN_PATH,
            val_dataset_path=TEST_PATH,
            output_dir=ARTIFACT_DIR,
            sequence_length=SEQUENCE_LENGTH,
            batch_size=BATCH_SIZE,
            epochs=FULL_TRAIN_EPOCHS,
            learning_rate=LEARNING_RATE,
            max_samples=FULL_TRAIN_MAX_SAMPLES,
            dry_run=False,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_monitor=EARLY_STOPPING_MONITOR,
        )
        print('Full training completed.')
        best_val_loss = min(history.history.get('val_loss', [float('nan')]))
        best_val_mse = min(history.history.get('val_severity_output_mse', [float('nan')]))
        best_val_ssim = max(history.history.get('val_severity_output_average_ssim', [float('nan')]))
        print({
            'best_val_loss': best_val_loss,
            'best_val_severity_mse': best_val_mse,
            'best_val_severity_ssim': best_val_ssim,
        })
    except Exception as exc:
        print('Full training failed.')
        traceback.print_exc()
        raise exc
else:
    print('Full training is disabled. Enable RUN_FULL_TRAINING when you are ready for the real 20-epoch run.')

Full training is disabled. Enable RUN_FULL_TRAINING when you are ready for the real 20-epoch run.


In [29]:
# ── Experiment history across all past runs ───────────────────────────────────
run_dirs = sorted(ARTIFACT_DIR.glob('run_*/run_config.json'))
if run_dirs:
    records = []
    for cfg_path in run_dirs:
        try:
            cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
            row = {
                'run_id': cfg.get('run_id', cfg_path.parent.name),
                'timestamp': cfg.get('timestamp', '')[:19],
                'dry_run': cfg.get('dry_run', False),
                'lr': cfg.get('hyperparameters', {}).get('learning_rate', ''),
                'epochs_trained': cfg.get('results', {}).get('epochs_trained', ''),
                'best_epoch': cfg.get('results', {}).get('best_epoch', ''),
                'best_val_loss': cfg.get('results', {}).get('best_val_loss', float('nan')),
                'best_val_mse': cfg.get('results', {}).get('best_val_severity_mse', float('nan')),
                'best_val_ssim': cfg.get('results', {}).get('best_val_severity_ssim', float('nan')),
                'best_val_acc': cfg.get('results', {}).get('best_val_identity_accuracy', float('nan')),
            }
            records.append(row)
        except Exception:
            pass
    print(f'Experiment history ({len(records)} runs):')
    print(pd.DataFrame(records).to_string(index=False))
else:
    print('No completed training runs found yet in', ARTIFACT_DIR)


No completed training runs found yet in C:\Users\Pranjal\Documents\GitHub\fifth-season\artifacts\notebook_training


## 6. Evaluation and Visualization
These functions compare the multi-task model against the known failure mode from the paper: over-smoothing and weak edge preservation. Keep this section off until the dry-run training cell succeeds.

In [30]:
if RUN_EVALUATION:
    metrics = evaluate_model(
        model_path=ARTIFACT_DIR / 'best_model.keras',
        test_dataset_path=TEST_PATH,
        history_path=ARTIFACT_DIR / 'history.json',
        output_dir=ARTIFACT_DIR / 'evaluation',
        sequence_length=SEQUENCE_LENGTH,
        max_samples=EVALUATION_MAX_SAMPLES,
    )
    print('Evaluation metrics:')
    print(json.dumps(metrics, indent=2))
else:
    print('Evaluation is disabled. Turn it on after you have a trained model artifact in the training output directory.')

if history is not None:
    fig, axs = plt.subplots(1, 3, figsize=(16, 4))

    axs[0].plot(history.history.get('loss', []), label='Train Loss')
    axs[0].plot(history.history.get('val_loss', []), label='Val Loss')
    axs[0].set_title('Overall Loss')
    axs[0].legend()

    axs[1].plot(history.history.get('severity_output_mse', []), label='Train Severity MSE')
    axs[1].plot(history.history.get('val_severity_output_mse', []), label='Val Severity MSE')
    axs[1].set_title('Severity MSE')
    axs[1].legend()

    train_ssim = history.history.get('severity_output_average_ssim', history.history.get('average_ssim', []))
    val_ssim = history.history.get('val_severity_output_average_ssim', history.history.get('val_average_ssim', []))
    axs[2].plot(train_ssim, label='Train Severity SSIM')
    axs[2].plot(val_ssim, label='Val Severity SSIM')
    axs[2].set_title('Severity SSIM')
    axs[2].legend()

    plt.tight_layout()
    plt.show()
else:
    print('No in-memory training history is available yet. Run either the dry-run cell or the full-training cell first.')


Evaluation metrics:
{
  "mse": 0.000758871011545553,
  "average_ssim": 0.9330116799649071,
  "num_windows": 68.0
}
No in-memory training history is available yet. Run either the dry-run cell or the full-training cell first.


## 7. Reading Failures
Use the failing cell to narrow the problem quickly:

- If raw inspection fails, the issue is in source files, dimensions, or coordinates.
- If Sentinel processing fails, inspect missing variables or malformed time axes in the Sentinel file.
- If MERRA-2 processing fails, inspect duplicate daily files, missing time metadata, or interpolation issues.
- If fusion fails, inspect timestamp overlap and whether both processed datasets share the same 291x512 grid.
- If generator checks fail, inspect feature metadata, NaNs, and whether the sequence length is larger than the available timeline.
- If the cropped model smoke test fails, the problem is model shape logic rather than memory pressure.
- If the cropped smoke test passes but full-resolution build or training fails, the problem is memory or runtime scale rather than tensor compatibility.